<a href="https://colab.research.google.com/github/Revanth3112/Group3_Capstone/blob/main/Group3_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size="10">Retail Sales Dashboard & Business Insights</font>

#Data Cleaning and Transformation

In [ ]:
#Loading the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
data = pd.read_csv("/content/sample_data/customer_shopping_data.csv")
data.head()

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon


In [ ]:
#shape of the dataset
data.shape

(99457, 10)

In [ ]:
#datatypes of the columns
data.dtypes

,0
invoice_no,object
customer_id,object
gender,object
age,int64
category,object
quantity,int64
price,float64
payment_method,object
invoice_date,object
shopping_mall,object


In [ ]:
#checking for missing values
data.isnull().sum()

,0
invoice_no,0
customer_id,0
gender,0
age,0
category,0
quantity,0
price,0
payment_method,0
invoice_date,0
shopping_mall,0


No missing values are there in the dataset.

In [ ]:
#checking for duplicated rows
data.duplicated().sum()

np.int64(0)

No duplicated rows are there in the dataset.

In [ ]:
#descriptive statistics
data.describe()

,age,quantity,price
count,99457.000000,99457.000000,99457.000000
mean,43.427089,3.003429,689.256321
std,14.990054,1.413025,941.184567
min,18.000000,1.000000,5.230000
25%,30.000000,2.000000,45.450000
50%,43.000000,3.000000,203.300000
75%,56.000000,4.000000,1200.320000
max,69.000000,5.000000,5250.000000


In [ ]:
#changing the datatype of invoice_date
data['invoice_date'] = pd.to_datetime(
    data['invoice_date'],
    dayfirst=True,
    errors='coerce'
)
data['invoice_date'].dtypes

dtype('<M8[ns]')

In [ ]:
#Feature Engineering
#Time breakdowns
data['year']        = data['invoice_date'].dt.year
data['month']       = data['invoice_date'].dt.month
data['month_name']  = data['invoice_date'].dt.strftime('%B')
data['day_of_week'] = data['invoice_date'].dt.strftime('%A')
data['quarter']     = data['invoice_date'].dt.quarter

#Revenue KPI
data['revenue'] = (data['price'] * data['quantity']).round(2)

#Age Segmentation
age_bins   = [17, 25, 35, 45, 55, 70]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-69']
data['age_group'] = pd.cut(data['age'], bins=age_bins, labels=age_labels)

#Revenue Segmentation
rev_bins   = [0, 50, 200, 1000, 3000, np.inf]
rev_labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
data['revenue_tier'] = pd.cut(data['revenue'], bins=rev_bins, labels=rev_labels)


In [ ]:
#shape of the data and displaying the columns after feature engineering
print("\nThe shape of the updated dataset")
print(data.shape)
print("\nThe first 10 rows of the dataset")
print(data.head(10))
print("\nNumber of Missing values")
print(data.isnull().sum().sum())


The shape of the updated dataset
(99457, 18)

The first 10 rows of the dataset
  invoice_no customer_id  gender  age   category  quantity    price  \
0    I138884     C241288  Female   28   Clothing         5  1500.40   
1    I317333     C111565    Male   21      Shoes         3  1800.51   
2    I127801     C266599    Male   20   Clothing         1   300.08   
3    I173702     C988172  Female   66      Shoes         5  3000.85   
4    I337046     C189076  Female   53      Books         4    60.60   
5    I227836     C657758  Female   28   Clothing         5  1500.40   
6    I121056     C151197  Female   49  Cosmetics         1    40.66   
7    I293112     C176086  Female   32   Clothing         2   600.16   
8    I293455     C159642    Male   69   Clothing         3   900.24   
9    I326945     C283361  Female   60   Clothing         2   600.16   

  payment_method invoice_date     shopping_mall  year  month month_name  \
0    Credit Card   2022-08-05            Kanyon  2022      8   

In [ ]:
#Downloading the cleaned dataset
from google.colab import files
data.to_excel('customer_shopping_cleaned.xlsx', index=False, sheet_name='Sales Data')
files.download('customer_shopping_cleaned.xlsx')
print("Excel file ready for Visualization!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Excel file ready for Visualization!


#SQL based KPI calculations

In [ ]:
#Setting up of SQLite
import sqlite3

#Register DataFrame as a SQL table
conn = sqlite3.connect(':memory:')
data.to_sql('sales', conn, index=False, if_exists='replace')

print("SQLite table 'sales' created in memory")
print(f"   Rows loaded : {len(data):,}")
print(f"   Columns     : {list(data.columns)}")

#Helper to run SQL and display results
def run_sql(query):
    return pd.read_sql_query(query, conn)

SQLite table 'sales' created in memory
   Rows loaded : 99,457
   Columns     : ['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity', 'price', 'payment_method', 'invoice_date', 'shopping_mall', 'year', 'month', 'month_name', 'day_of_week', 'quarter', 'revenue', 'age_group', 'revenue_tier']


In [ ]:
#overall business KPIs
kpi_overall = run_sql("""
    SELECT
        ROUND(SUM(revenue), 2)              AS total_revenue,
        COUNT(invoice_no)                   AS total_transactions,
        ROUND(AVG(revenue), 2)              AS avg_order_value,
        SUM(quantity)                       AS total_units_sold,
        COUNT(DISTINCT customer_id)         AS unique_customers,
        ROUND(SUM(revenue) /
              COUNT(DISTINCT customer_id), 2) AS revenue_per_customer
    FROM sales
""")

print("OVERALL BUSINESS KPIs")
kpi_overall

OVERALL BUSINESS KPIs


,total_revenue,total_transactions,avg_order_value,total_units_sold,unique_customers,revenue_per_customer
0,2.515058e+08,99457,2528.79,298712,99457,2528.79


In [ ]:
#Revenue by Category
kpi_category = run_sql("""
    SELECT
        category,
        COUNT(invoice_no)              AS transactions,
        SUM(quantity)                  AS units_sold,
        ROUND(SUM(revenue), 2)         AS total_revenue,
        ROUND(AVG(revenue), 2)         AS avg_order_value,
        ROUND(SUM(revenue) * 100.0 /
              (SELECT SUM(revenue) FROM sales), 2) AS revenue_pct
    FROM sales
    GROUP BY category
    ORDER BY total_revenue DESC
""")

print("REVENUE BY CATEGORY")
kpi_category

REVENUE BY CATEGORY


,category,transactions,units_sold,total_revenue,avg_order_value,revenue_pct
0,Clothing,34487,103558,1.139968e+08,3305.50,45.33
1,Shoes,10034,30217,6.655345e+07,6632.79,26.46
2,Technology,4996,15021,5.786235e+07,11581.74,23.01
3,Cosmetics,15097,45465,6.792863e+06,449.95,2.70
4,Toys,10087,30321,3.980426e+06,394.61,1.58
5,Food & Beverage,14776,44277,8.495351e+05,57.49,0.34
6,Books,4981,14982,8.345529e+05,167.55,0.33
7,Souvenir,4999,14871,6.358247e+05,127.19,0.25


In [ ]:
#Revenue by Shopping mall
kpi_mall = run_sql("""
    SELECT
        shopping_mall,
        COUNT(invoice_no)              AS transactions,
        ROUND(SUM(revenue), 2)         AS total_revenue,
        ROUND(AVG(revenue), 2)         AS avg_order_value,
        ROUND(SUM(revenue) * 100.0 /
              (SELECT SUM(revenue) FROM sales), 2) AS revenue_pct
    FROM sales
    GROUP BY shopping_mall
    ORDER BY total_revenue DESC
""")

print("REVENUE BY SHOPPING MALL")
kpi_mall

REVENUE BY SHOPPING MALL


,shopping_mall,transactions,total_revenue,avg_order_value,revenue_pct
0,Mall of Istanbul,19943,50872481.68,2550.89,20.23
1,Kanyon,19823,50554231.10,2550.28,20.10
2,Metrocity,15011,37302787.33,2485.03,14.83
3,Metropol AVM,10161,25379913.19,2497.78,10.09
4,Istinye Park,9781,24618827.68,2517.01,9.79
5,Zorlu Center,5075,12901053.82,2542.08,5.13
6,Cevahir AVM,4991,12645138.20,2533.59,5.03
7,Viaport Outlet,4914,12521339.72,2548.10,4.98
8,Emaar Square Mall,4811,12406100.29,2578.69,4.93
9,Forum Istanbul,4947,12303921.24,2487.15,4.89


In [ ]:
#Revenue by gender
kpi_gender = run_sql("""
    SELECT
        gender,
        COUNT(invoice_no)              AS transactions,
        SUM(quantity)                  AS units_sold,
        ROUND(SUM(revenue), 2)         AS total_revenue,
        ROUND(AVG(revenue), 2)         AS avg_order_value,
        ROUND(SUM(revenue) * 100.0 /
              (SELECT SUM(revenue) FROM sales), 2) AS revenue_pct
    FROM sales
    GROUP BY gender
    ORDER BY total_revenue DESC
""")

print("REVENUE BY GENDER")
kpi_gender

REVENUE BY GENDER


,gender,transactions,units_sold,total_revenue,avg_order_value,revenue_pct
0,Female,59482,178659,1.502071e+08,2525.25,59.72
1,Male,39975,120053,1.012987e+08,2534.05,40.28


In [ ]:
#Revenue by payment method
kpi_payment = run_sql("""
    SELECT
        payment_method,
        COUNT(invoice_no)              AS transactions,
        ROUND(SUM(revenue), 2)     AS total_revenue,
        ROUND(AVG(revenue), 2)     AS avg_order_value,
        ROUND(SUM(revenue) * 100.0 /
              (SELECT SUM(revenue) FROM sales), 2) AS revenue_pct
    FROM sales
    GROUP BY payment_method
    ORDER BY total_revenue DESC
""")

print("REVENUE BY PAYMENT METHOD")
kpi_payment

REVENUE BY PAYMENT METHOD


,payment_method,transactions,total_revenue,avg_order_value,revenue_pct
0,Cash,44447,1.128322e+08,2538.58,44.86
1,Credit Card,34931,8.807712e+07,2521.46,35.02
2,Debit Card,20079,5.059643e+07,2519.87,20.12


In [ ]:
#Revenue by age group
kpi_age = run_sql("""
    SELECT
        age_group,
        COUNT(invoice_no)              AS transactions,
        SUM(quantity)                  AS units_sold,
        ROUND(SUM(revenue), 2)         AS total_revenue,
        ROUND(AVG(revenue), 2)         AS avg_order_value,
        ROUND(SUM(revenue) * 100.0 /
              (SELECT SUM(revenue) FROM sales), 2) AS revenue_pct
    FROM sales
    GROUP BY age_group
    ORDER BY age_group
""")

print("REVENUE BY AGE GROUP")
kpi_age

REVENUE BY AGE GROUP


,age_group,transactions,units_sold,total_revenue,avg_order_value,revenue_pct
0,18-25,15359,46028,38118271.35,2481.82,15.16
1,26-35,19059,57265,47879659.58,2512.18,19.04
2,36-45,19436,58483,50184235.79,2582.02,19.95
3,46-55,19016,57086,48219742.64,2535.75,19.17
4,56-69,26587,79850,67103884.89,2523.94,26.68


In [ ]:
#Revenue by quarter
kpi_quarter = run_sql("""
    SELECT
        year,
        'Q' || quarter               AS quarter,
        COUNT(invoice_no)            AS transactions,
        ROUND(SUM(revenue), 2)       AS quarterly_revenue,
        ROUND(AVG(revenue), 2)       AS avg_order_value
    FROM sales
    GROUP BY year, quarter
    ORDER BY year, quarter
""")

print("QUARTERLY REVENUE")
kpi_quarter

QUARTERLY REVENUE


,year,quarter,transactions,quarterly_revenue,avg_order_value
0,2021,Q1,11055,27869289.22,2520.97
1,2021,Q2,11355,28447569.86,2505.29
2,2021,Q3,11377,29129941.00,2560.42
3,2021,Q4,11595,29113770.51,2510.89
4,2022,Q1,11241,28095108.22,2499.34
5,2022,Q2,11410,28921222.52,2534.73
6,2022,Q3,11488,29326937.83,2552.83
7,2022,Q4,11412,29093545.51,2549.38
8,2023,Q1,8524,21508409.58,2523.28


In [ ]:
#Revenue by day_of_week
kpi_dow = run_sql("""
    SELECT
        day_of_week,
        COUNT(invoice_no)            AS transactions,
        ROUND(SUM(revenue), 2)   AS total_revenue,
        ROUND(AVG(revenue), 2)   AS avg_order_value
    FROM sales
    GROUP BY day_of_week
    ORDER BY total_revenue DESC
""")

print("REVENUE BY DAY OF WEEK")
kpi_dow

REVENUE BY DAY OF WEEK


,day_of_week,transactions,total_revenue,avg_order_value
0,Monday,14383,37296648.11,2593.11
1,Tuesday,14217,36298096.66,2553.15
2,Thursday,14129,35738148.63,2529.42
3,Friday,14347,35728331.39,2490.30
4,Sunday,14140,35689090.61,2523.98
5,Wednesday,14120,35575650.13,2519.52
6,Saturday,14121,35179828.72,2491.31


In [ ]:
#category x gender analysis
kpi_cat_gender = run_sql("""
    SELECT
        category,
        gender,
        COUNT(invoice_no)            AS transactions,
        ROUND(SUM(revenue), 2)       AS total_revenue,
        ROUND(AVG(revenue), 2)       AS avg_order_value
    FROM sales
    GROUP BY category, gender
    ORDER BY category, gender
""")

print("CATEGORY × GENDER")
kpi_cat_gender

CATEGORY × GENDER


,category,gender,transactions,total_revenue,avg_order_value
0,Books,Female,2906,489314.70,168.38
1,Books,Male,2075,345238.20,166.38
2,Clothing,Female,20652,68251695.60,3304.85
3,Clothing,Male,13835,45745095.44,3306.48
4,Cosmetics,Female,9070,4066772.54,448.38
5,Cosmetics,Male,6027,2726090.36,452.31
6,Food & Beverage,Female,8804,505322.60,57.40
7,Food & Beverage,Male,5972,344212.45,57.64
8,Shoes,Female,5967,39425167.30,6607.20
9,Shoes,Male,4067,27128284.17,6670.34


#Exploratory Data Analysis

In [ ]:
print('----------- DATASET OVERVIEW -----------')
print(f'Total Transactions   : {len(data):,}')
print(f'Unique Customers     : {data["customer_id"].nunique():,}')
print(f'Unique Invoices      : {data["invoice_no"].nunique():,}')
print(f'Shopping Malls       : {data["shopping_mall"].nunique()}')
print(f'Product Categories   : {data["category"].nunique()}')
print(f'Date Range           : {data["invoice_date"].min().date()} to {data["invoice_date"].max().date()}')
print(f'Total Revenue        : ₺{data["revenue"].sum():,.2f}')
print(f'Avg Order Value      : ₺{data["revenue"].mean():,.2f}')
print(f'Avg Items per Order  : {data["quantity"].mean():.2f}')

----------- DATASET OVERVIEW -----------
Total Transactions   : 99,457
Unique Customers     : 99,457
Unique Invoices      : 99,457
Shopping Malls       : 10
Product Categories   : 8
Date Range           : 2021-01-01 to 2023-03-08
Total Revenue        : ₺251,505,794.25
Avg Order Value      : ₺2,528.79
Avg Items per Order  : 3.00


In [ ]:
gender_counts = data['gender'].value_counts()
print('Gender Split:')
for g, cnt in gender_counts.items():
    print(f'  {g:>6}: {cnt:,}  ({cnt/len(data)*100:.1f}%)')

Gender Split:
  Female: 59,482  (59.8%)
    Male: 39,975  (40.2%)


In [ ]:
print('Revenue by Category:')
cat_rev = data.groupby('category')['revenue'].sum().sort_values(ascending=False)
for cat, rev in cat_rev.items():
    pct = rev / cat_rev.sum() * 100
    bar = '█' * int(pct / 2)
    print(f'  {cat:<20} ₺{rev:>15,.0f}  ({pct:5.1f}%)  {bar}')

Revenue by Category:
  Clothing             ₺    113,996,791  ( 45.3%)  ██████████████████████
  Shoes                ₺     66,553,451  ( 26.5%)  █████████████
  Technology           ₺     57,862,350  ( 23.0%)  ███████████
  Cosmetics            ₺      6,792,863  (  2.7%)  █
  Toys                 ₺      3,980,426  (  1.6%)  
  Food & Beverage      ₺        849,535  (  0.3%)  
  Books                ₺        834,553  (  0.3%)  
  Souvenir             ₺        635,825  (  0.3%)  


In [ ]:
customer_visits = data.groupby('customer_id')['invoice_no'].nunique()
repeat_customers = (customer_visits > 1).sum()
total_customers  = len(customer_visits)

print(f'Total unique customers       : {total_customers:,}')
print(f'Repeat customers (>1 visit)  : {repeat_customers:,}  ({repeat_customers/total_customers*100:.1f}%)')
print(f'Avg visits per customer      : {customer_visits.mean():.2f}')
print()
print('Visit frequency (how many times each customer shopped):')
print(customer_visits.value_counts().sort_index().head(10))

Total unique customers       : 99,457
Repeat customers (>1 visit)  : 0  (0.0%)
Avg visits per customer      : 1.00

Visit frequency (how many times each customer shopped):
invoice_no
1    99457
Name: count, dtype: int64


### Statistical Measures

In [ ]:
print('STATISTICAL ANALYSIS: QUANTITY & TOTAL AMOUNT')
print(data.columns.tolist())
for col in ['quantity', 'revenue']:
    s = data[col]
    print(f'\n{col.upper()}')
    print(f'  Mean       : {s.mean():>12,.2f}')
    print(f'  Median     : {s.median():>12,.2f}')
    print(f'  Std Dev    : {s.std():>12,.2f}')
    print(f'  Variance   : {s.var():>12,.2f}')
    print(f'  Skewness   : {s.skew():>12.3f}  (>0 = right skewed, more low values)')
    print(f'  Kurtosis   : {s.kurt():>12.3f}  (>3 = heavy tails / outliers present)')
    print(f'  Min / Max  : {s.min():>10,.2f}  /  {s.max():,.2f}')
    print(f'  IQR (25–75): {s.quantile(0.75)-s.quantile(0.25):>12,.2f}')


print('\nCORRELATION: quantity vs total_amount')
corr = data['quantity'].corr(data['revenue'])
print(f'  Pearson r  : {corr:>12.3f}  (1=perfect positive, 0=no relation)')

STATISTICAL ANALYSIS: QUANTITY & TOTAL AMOUNT
['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity', 'price', 'payment_method', 'invoice_date', 'shopping_mall', 'year', 'month', 'month_name', 'day_of_week', 'quarter', 'revenue', 'age_group', 'revenue_tier']

QUANTITY
  Mean       :         3.00
  Median     :         3.00
  Std Dev    :         1.41
  Variance   :         2.00
  Skewness   :       -0.001  (>0 = right skewed, more low values)
  Kurtosis   :       -1.296  (>3 = heavy tails / outliers present)
  Min / Max  :       1.00  /  5.00
  IQR (25–75):         2.00

REVENUE
  Mean       :     2,528.79
  Median     :       600.17
  Std Dev    :     4,222.48
  Variance   : 17,829,301.72
  Skewness   :        2.871  (>0 = right skewed, more low values)
  Kurtosis   :       10.312  (>3 = heavy tails / outliers present)
  Min / Max  :       5.23  /  26,250.00
  IQR (25–75):     2,564.37

CORRELATION: quantity vs total_amount
  Pearson r  :        0.461  (1=perfect positi

In [ ]:
# BY PRODUCT CATEGORY
print('MEAN & VARIANCE BY PRODUCT CATEGORY\n')

summary = data.groupby('category')['revenue'].agg(
    Mean='mean',
    Variance='var',
    Std_Dev='std',
    Count='count'
).round(2)

print(summary)

MEAN & VARIANCE BY PRODUCT CATEGORY

                     Mean     Variance  Std_Dev  Count
category                                              
Books              167.55     17400.09   131.91   4981
Clothing          3305.50   6746510.76  2597.40  34487
Cosmetics          449.95    123815.04   351.87  15097
Food & Beverage     57.49      2059.85    45.39  14776
Shoes             6632.79  26844457.88  5181.16  10034
Souvenir           127.19     10248.80   101.24   4999
Technology       11581.74  82274186.36  9070.51   4996
Toys               394.61     94559.39   307.51  10087


In [ ]:
# BY PAYMENT METHOD
print('MEAN & VARIANCE BY PAYMENT METHOD\n')
summary1 = data.groupby('payment_method')['revenue'].agg(
    Mean='mean',
    Variance='var',
    Std_Dev='std',
    Count='count'
).round(2)
print(summary1)

MEAN & VARIANCE BY PAYMENT METHOD

                   Mean     Variance  Std_Dev  Count
payment_method                                      
Cash            2538.58  17935584.73  4235.04  44447
Credit Card     2521.46  17767725.92  4215.18  34931
Debit Card      2519.87  17702541.70  4207.44  20079


In [ ]:
# BY STORE TYPE
print('MEAN & VARIANCE BY STORE TYPE\n')
summary2 = data.groupby('shopping_mall')['revenue'].agg(
    Mean='mean',
    Variance='var',
    Std_Dev='std',
    Count='count'
).round(2)
print(summary2)

MEAN & VARIANCE BY STORE TYPE



NameError: name 'data' is not defined